#### Importar dados do PurpleAir

##### 1. Página guia de como usar a API https://api.purpleair.com/#api-sensors-get-sensors-data
##### 2. Criar conta com e-mail Google https://develop.purpleair.com/dashboards/organization 
##### 3. Criar API key para substituir no código e ter acesso às informaçôes

##### Baixar dados API PurpleAir

In [29]:
import os, time, math, requests, pandas as pd
from datetime import datetime, timedelta, timezone

In [30]:
API_KEY = "CC7DC672-9555-11F0-BDE5-4201AC1DC121"  # put your PurpleAir Read key here
assert API_KEY and API_KEY != "YOUR_READ_KEY_HERE", "Add your PurpleAir API key"

In [31]:
BASE = "https://api.purpleair.com/v1"
HEADERS = {"X-API-Key": API_KEY}

In [32]:
# Brazil bounding box
BBOX = dict(
    nwlng = -73.990556,  # westmost
    nwlat =  5.271944,   # northmost
    selng = -34.792778,  # eastmost
    selat = -33.751944   # southmost
)

In [33]:
FIELDS = ",".join([
    "sensor_index","name","latitude","longitude", "voc","ozone1","pm1.0",
    "pm2.5","pm10.0","date_created", "last_seen","location_type", "private", "model", "altitude"
])

In [34]:
def fetch_sensors_in_bbox(fields, bbox, limit=1000, page=1):
    params = dict(fields=fields, **bbox, limit=limit, page=page)
    r = requests.get(f"{BASE}/sensors", headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [35]:
resp = fetch_sensors_in_bbox(FIELDS, BBOX, limit=10000, page=1)

In [36]:
cols = resp.get("fields", [])
rows = resp.get("data", [])
df = pd.DataFrame(rows, columns=cols)

In [37]:
df

,sensor_index,date_created,last_seen,private,name,location_type,model,latitude,longitude,altitude,voc,ozone1,pm1.0,pm2.5,pm10.0
0,262813,1738289807,1758821801,0,PELD-TANG,0,PA-II-SD,-13.062653,-52.380970,1190,NaN,None,9.3,14.4,14.9
1,274372,1744987703,1758821857,0,Casa,1,PA-II-ZEN,-33.359848,-70.676420,1634,62.0,None,14.8,19.5,22.5
2,275450,1745526979,1758821755,0,Vielas da Água Preta,0,PA-II-ZEN,-23.536737,-46.692394,2435,77.0,None,4.3,5.7,7.8
3,278965,1747249329,1758821757,0,DIAMANTINO QUALIDADE DO AR,0,PA-II-ZEN,-14.399174,-56.436085,950,90.0,None,7.7,9.2,9.6
4,280308,1747933787,1758821830,0,El Monte,0,PA-II,-33.673256,-70.986206,962,NaN,None,15.6,22.9,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274,246453,1728317459,1758821835,0,GEOHealth_AREQ_Yura,0,PA-II-FLEX,-16.250816,-71.680020,8526,58.0,None,3.7,5.9,10.6
275,246457,1728317470,1758412835,0,GEOHealth_CUS_CcapacmarcaOefa3,0,PA-II-FLEX,-14.008719,-72.001175,11774,194.0,None,6.6,9.5,10.3
276,247259,1729002779,1758219604,0,RIO401101,0,PA-II,-22.911589,-43.756820,2,NaN,None,1.9,3.8,5.1
277,251937,1732137270,1758821520,0,"Universidad Privada Boliviana, Campus Achocalla",0,PA-II-FLEX,-16.575330,-68.126945,11548,106.0,None,0.7,1.1,1.4


##### Filtrar apenas latitudes e longitudes dentro do polígno do Brasil

In [38]:
def df_brazil_only(df, lat_col="latitude", lon_col="longitude", drop_coords=True):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326"
    )
    # alterar colunas de datas para datetime 
    df["date_created"] = pd.to_datetime(df["date_created"], unit="s", utc=True)
    df["date_created"]  = df["date_created"].dt.tz_convert("America/Sao_Paulo")
    df["last_seen"] = pd.to_datetime(df["last_seen"], unit="s", utc=True)
    df["last_seen"]  = df["last_seen"].dt.tz_convert("America/Sao_Paulo")
    
    # polígono do país para extrair contornos
    url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    world = gpd.read_file(url)
    brazil_poly = world.loc[world["SOVEREIGNT"] == "Brazil", "geometry"].iloc[0]

    # cria mask para manter somente lat e long dentro do contorno
    mask = gdf.within(brazil_poly)
    gdf = gdf[mask].copy()
    
    return gdf

In [39]:
df_br = df_brazil_only(df)   # df_br contém somente as linhas lat e long dentro do Brasil
print("Shape sensores PurpleAir no Brasil: " +str(df_br.shape))
df_br.head()

NameError: name 'gpd' is not defined

In [ ]:
df_br['start_year'] = df_br['date_created'].dt.year
df_br['end_year'] = df_br['last_seen'].dt.year

def get_years_in_range(row): # extrair todos os anos de medição de cada sensor
    return list(range(row['start_year'], row['end_year'] + 1))

df_br['years_in_range'] = df_br.apply(get_years_in_range, axis=1)

def years_to_list(v):
    if isinstance(v, (list, tuple, set)):
        return ",".join(str(x) for x in v)              
    return v                                            

df_br["years_in_range"] = df_br["years_in_range"].apply(years_to_list)

In [ ]:
import numpy as np

In [ ]:
cols = ["voc", "ozone1", "pm1.0", "pm2.5", "pm10.0"]  
labels = {"voc":"VOC", "pm1.0":"PM1", "pm2.5":"PM25", "pm10.0":"PM10", "ozone1":"OZONE"}

def get_pollutants(df_br, cols, labels):
    df_br["pollutants_all"] = df_br[cols].apply(
    lambda r: ",".join(labels[c] for c in cols if r[c] is not None) or pd.NA,
    axis=1
    )

    return df_br

In [ ]:
df_br = get_pollutants(df_br, cols, labels)
df_br.head()

##### Extrair nome da cidade e estado baseado na latitude e longitude

In [ ]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
from geopy.extra.rate_limiter import RateLimiter

In [ ]:
geolocator = Nominatim(user_agent="bixtech-airquality", timeout=10)
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1.1, max_retries=2, error_wait_seconds=2)

In [ ]:
def get_city_name(latitude, longitude):
    try:
        location = geolocator.reverse((latitude, longitude), exactly_one=True)
        if location and location.address:
            address_parts = location.raw.get('address', {})
            return address_parts.get('city')
        return None
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        print(f"Error geocoding {latitude}, {longitude}: {e}")
        return None

In [ ]:
def get_uf_name(latitude, longitude):
    try:
        location = geolocator.reverse((latitude, longitude), exactly_one=True)
        if location and location.address:
            address_parts = location.raw.get('address', {})
            return address_parts.get('state')
        return None
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        print(f"Error geocoding {latitude}, {longitude}: {e}")
        return None

In [ ]:
df_br['CIDADE'] = df_br.apply(lambda row: get_city_name(row['latitude'], row['longitude']), axis=1)
df_br['UF'] = df_br.apply(lambda row: get_uf_name(row['latitude'], row['longitude']), axis=1)

In [ ]:
df_br.head()

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import folium

m = folium.Map(location=[-14.235, -51.925], zoom_start=4)

for lat, lon in zip(df_br["latitude"], df_br["longitude"]):
    if pd.notna(lat) and pd.notna(lon):
        folium.CircleMarker(location=[lat, lon], radius=3).add_to(m)

In [ ]:
m

##### Transformar dataframe conforme colunas do Monitoramento_QAr_BR

In [ ]:
campos = ["sensor_index", "UF","CIDADE","CD_MUN","ID_OEMA","ID_MMA","ID_MMA_COMPLETO","PROPRIETARIO",
          "PROP_ENTIDADE","OPERADOR","OP_ENTIDADE","FUNCIONAMENTO","CATEGORIA","METODO",
          "CALIBRACAO","MARCA","MODELO","POLUENTE","COD_POLUENTE","MOBILIDADE","REP_ESPACIAL",
          "FINALIDADE","STATUS","INICIO","FIM","LATITUDE","LONGITUDE","MONITORAR","FONTE",
          "CERTIFICACAO","COD_UF_IBGE","ANOS_MONITORADOS","BASE_DADOS","ELEVACAO"]

In [ ]:
df_purple = pd.DataFrame(columns=campos) 

In [ ]:
manual_map = {
    "latitude":  "LATITUDE",
    "longitude": "LONGITUDE",
    "name": "ID_OEMA",
    "private": "PROP_ENTIDADE",
    "model": "MODELO",
    "years_in_range": "ANOS_MONITORADOS",
    "pollutants_all": "POLUENTE",
    "date_created": "INICIO",
    "last_seen": "FIM",
    "altitude": "ELEVACAO"
}

df_br = df_br.rename(columns=manual_map)

In [ ]:
df_purple = df_purple.reindex(index=df_br.index)  
common = [c for c in campos if c in df_br.columns]
df_purple.loc[:, common] = df_br[common].values
df_purple.head()

In [ ]:
defaults = {
    "FUNCIONAMENTO": "Automatica",
    "CATEGORIA":     "Indicativa",
    "MARCA":         "PurpleAir",
    "FONTE":         "PurpleAir 2025",
}

df_purple = df_purple.fillna(value=defaults)

In [ ]:
map_prop = {
    0: "Publico", 1: "Privada",
    False: "Publico", True: "Privado",
    "0": "Publico", "1": "Privado",
}

df_purple["PROP_ENTIDADE"] = df_purple["PROP_ENTIDADE"].replace(map_prop)
df_purple["PROP_ENTIDADE"] = df_purple["PROP_ENTIDADE"].astype("string")

In [ ]:
import unicodedata

In [ ]:
def _key(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    s = unicodedata.normalize("NFKD", s).encode("ASCII","ignore").decode("ASCII")
    return s.lower()

In [ ]:
# mapa: nome da UF -> sigla
_name_to_sigla = {
    "acre":"AC","alagoas":"AL","amapa":"AP","amazonas":"AM","bahia":"BA","ceara":"CE",
    "distrito federal":"DF","espirito santo":"ES","goias":"GO","maranhao":"MA",
    "mato grosso":"MT","mato grosso do sul":"MS","minas gerais":"MG","para":"PA",
    "paraiba":"PB","parana":"PR","pernambuco":"PE","piaui":"PI","rio de janeiro":"RJ",
    "rio grande do norte":"RN","rio grande do sul":"RS","rondonia":"RO","roraima":"RR",
    "santa catarina":"SC","sao paulo":"SP","sergipe":"SE","tocantins":"TO"
}

# também aceitar quando a coluna já vier com a sigla
for sig in list(_name_to_sigla.values()):
    _name_to_sigla[sig.lower()] = sig

# códigos IBGE por sigla
_sigla_to_ibge = {
    "AC":"12","AL":"27","AP":"16","AM":"13","BA":"29","CE":"23","DF":"53","ES":"32",
    "GO":"52","MA":"21","MT":"51","MS":"50","MG":"31","PA":"15","PB":"25","PR":"41",
    "PE":"26","PI":"22","RN":"24","RS":"43","RJ":"33","RO":"11","RR":"14","SC":"42",
    "SP":"35","SE":"28","TO":"17"
}

In [ ]:
# substituir nome por sigla na coluna UF
df_purple["UF"] = df_purple["UF"].apply(lambda x: _name_to_sigla.get(_key(x), pd.NA))

# preencher COD_UF_IBGE a partir da sigla
df_purple["COD_UF_IBGE"] = df_purple["UF"].map(_sigla_to_ibge)

# dtypes opcionais
df_purple["UF"] = df_purple["UF"].astype("string")
df_purple["COD_UF_IBGE"] = df_purple["COD_UF_IBGE"].astype("string")

In [ ]:
df_purple.head()

In [ ]:
df_purple.groupby(["UF"]).count()

In [ ]:
# from pathlib import Path

# base = Path.cwd().parent  # .../RQAR_2025_book
# out_dir = base / "data" / "DADOS_ESTACOES"
# out_dir.mkdir(parents=True, exist_ok=True)

# out_file = out_dir / "Compiled_PurpleAirStations.csv"
# df_purple.to_csv(out_file, index=False, encoding="utf-8")
# print("Saved to:", out_file.resolve())

##### Salvar um arquivo .csv por estado com os dados coletados via API PurpleAir

In [ ]:
base = Path.cwd().parent  # .../RQAR_2025_book
out_dir = base / "data" / "DADOS_ESTACOES" / "UFs_indicativas"
out_dir.mkdir(parents=True, exist_ok=True)

def save_ufs_files(df_purple, uf_col="UF", base=base, out_dir=out_dir):
    dfx = df_purple.copy()

    for uf, sub in dfx.groupby(uf_col, dropna=False):
        #safe_uf = "".join(ch if ch.isalnum() or ch in (" ", "-", "_") else "_" for ch in str(uf))
        name_template=f"{uf}_indicativas.csv"
        fname = name_template.format(uf)
        sub.to_csv(out_dir / fname, index=False)

save_ufs_files(df_purple, uf_col="UF", base=base, out_dir=out_dir)

#### Comparar dados das UFs com base de dados PurpleAir

In [ ]:
base = Path.cwd().parent  
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

df_mma = pd.read_csv(out_dir / "Monitoramento_QAr_BR.csv")
df_mma.columns

In [ ]:
id_col = "ID_OEMA"

def norm(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.upper()

# normalized ID columns
mma_id = norm(df_mma[id_col])
pur_id = norm(df_purple[id_col])

# IDs present in both dataframes
common_ids = pd.Index(mma_id.dropna()).intersection(pur_id.dropna())
print("IDs present in both:", len(common_ids))

# rows involved on each side
mma_overlap = df_mma[mma_id.isin(common_ids)]
pur_overlap = df_purple[pur_id.isin(common_ids)]
print("Rows in df_mma with overlapping ID_OEMA:", len(mma_overlap))
print("Rows in df_purple_exp with overlapping ID_OEMA:", len(pur_overlap))

# summary count per ID showing how many times it appears in each df
both = pd.concat(
    [
        pd.DataFrame({id_col: mma_id, "_src": "mma"}),
        pd.DataFrame({id_col: pur_id, "_src": "purple"}),
    ],
    ignore_index=True,
)
summary = (
    both.dropna(subset=[id_col])
        .groupby([id_col, "_src"]).size()
        .unstack(fill_value=0)
        .query("mma > 0 and purple > 0")
        .sort_index()
)
print("Overlapping unique IDs:", summary.shape[0])

##### Overlap de mapas - Coordenadas MMA e PurpleAir

In [ ]:
def add_points(df, lat_col="latitude", lon_col="longitude", color="blue", name="Layer"):
    fg = folium.FeatureGroup(name=name, show=True)
    for lat, lon in zip(df[lat_col], df[lon_col]):
        if pd.notna(lat) and pd.notna(lon):
            folium.CircleMarker(
                location=[lat, lon],
                radius=3,
                color=color,
                fill=True,
                fill_opacity=0.8,
                opacity=0.8
            ).add_to(fg)
    fg.add_to(m)

# Plot your two dataframes
add_points(df_purple,  lat_col="LATITUDE", lon_col="LONGITUDE", color="blue",  name="PurpleAir")
add_points(df_mma, lat_col="LATITUDE", lon_col="LONGITUDE", color="red",   name="MMA")

# Layer control to toggle visibility
folium.LayerControl(collapsed=False).add_to(m)

m  

In [ ]:
def plot_purple_mma_map(df_purple, df_mma, lat_col="LATITUDE", lon_col="LONGITUDE", uf_col="UF",
                    buffer_m=100, draw_buffers=False):
    # Clean coords
    df_purple  = df_purple.dropna(subset=[lat_col, lon_col]).copy()
    df_mma = df_mma.dropna(subset=[lat_col, lon_col]).copy()

    # GeoDataFrames in WGS84
    gdf_purple = gpd.GeoDataFrame(
        df_purple,
        geometry=gpd.points_from_xy(df_purple[lon_col], df_purple[lat_col]),
        crs="EPSG:4326"
    )
    gdf_mma = gpd.GeoDataFrame(
        df_mma,
        geometry=gpd.points_from_xy(df_mma[lon_col], df_mma[lat_col]),
        crs="EPSG:4326"
    )

    # Project to meters for accurate buffers
    gdf_purple_m  = gdf_purple.to_crs("EPSG:5880")
    gdf_mma_m = gdf_mma.to_crs("EPSG:5880")

    # Build buffers
    br_buf  = gdf_purple_m.copy()
    br_buf["geometry"] = br_buf.geometry.buffer(buffer_m)
    mma_buf = gdf_mma_m.copy()
    mma_buf["geometry"] = mma_buf.geometry.buffer(buffer_m)

    # Cross matches only
    match_br  = gpd.sjoin(br_buf[["geometry"]],  gdf_mma_m[["geometry"]], predicate="contains", how="inner")
    match_mma = gpd.sjoin(mma_buf[["geometry"]], gdf_purple_m[["geometry"]],  predicate="contains", how="inner")

    overlap_br_idx  = match_br.index.unique()
    overlap_mma_idx = match_mma.index.unique()

    gdf_purple_m["overlap_cross"]  = gdf_purple_m.index.isin(overlap_br_idx)
    gdf_mma_m["overlap_cross"] = gdf_mma_m.index.isin(overlap_mma_idx)

    # Back to lat lon for map
    gdf_purple_ll  = gdf_purple_m.to_crs("EPSG:4326")
    gdf_mma_ll = gdf_mma_m.to_crs("EPSG:4326")

    # Map
    m = folium.Map(location=[-14.235, -51.925], zoom_start=4, tiles="CartoDB positron")

    # Optionally draw buffer outlines
    if draw_buffers:
        for _, r in gdf_purple_ll.iterrows():
            folium.Circle(
                location=[r.geometry.y, r.geometry.x],
                radius=buffer_m,
                color="red",
                fill=False,
                opacity=0.3
            ).add_to(m)
        for _, r in gdf_mma_ll.iterrows():
            folium.Circle(
                location=[r.geometry.y, r.geometry.x],
                radius=buffer_m,
                color="blue",
                fill=False,
                opacity=0.3
            ).add_to(m)

    # Points: blue for BR, red for MMA, yellow if cross overlap
    for _, r in gdf_purple_ll.iterrows():
        lat, lon = r.geometry.y, r.geometry.x
        folium.CircleMarker(
            [lat, lon],
            radius=4,
            color="yellow" if r["overlap_cross"] else "blue",
            fill=True, fill_opacity=0.9, opacity=0.9,
            popup=f"br | {lat:.6f}, {lon:.6f}"
        ).add_to(m)

    for _, r in gdf_mma_ll.iterrows():
        lat, lon = r.geometry.y, r.geometry.x
        folium.CircleMarker(
            [lat, lon],
            radius=4,
            color="yellow" if r["overlap_cross"] else "red",
            fill=True, fill_opacity=0.9, opacity=0.9,
            popup=f"mma | {lat:.6f}, {lon:.6f}"
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    # Table of yellow points
    yellow_br  = gdf_purple_ll.loc[gdf_purple_ll["overlap_cross"], ["geometry", "UF", "ID_OEMA"]].copy()
    yellow_mma  = gdf_purple_ll.loc[gdf_purple_ll["overlap_cross"], ["geometry", "UF", "ID_OEMA"]].copy()    
    
    yellow_br  = yellow_br.assign(
        source="br",
        latitude=yellow_br.geometry.y,
        longitude=yellow_br.geometry.x,
        uf="UF",
        oema="ID_OEMA"
    )[["source","latitude","longitude","UF"]]
    
    yellow_mma = yellow_mma.assign(
        source="mma",
        latitude=yellow_mma.geometry.y,
        longitude=yellow_mma.geometry.x,
        uf="UF",
        oema="ID_OEMA"
    )[["source","latitude","longitude","UF","ID_OEMA"]]
    
    frames = [f for f in (yellow_br, yellow_mma) if not f.empty]
    yellow = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
        columns=["source","latitude","longitude","UF","ID_OEMA"]
    )

    # 1) Join BR buffers to MMA points -> all MMA points within 100 m of each BR point
    br_buf_i = br_buf[["geometry"]].reset_index().rename(columns={"index": "br_idx"})
    mma_pts  = gdf_mma_m.copy()
    mma_pts["mma_idx"] = mma_pts.index
    
    pairs_m = gpd.sjoin(
        br_buf_i, 
        mma_pts[["geometry", "mma_idx", *(["UF"] if "UF" in mma_pts.columns else []), *(["ID_OEMA"] if "ID_OEMA" in mma_pts.columns else [])]],
        predicate="contains",
        how="inner"
    )
    
    # 2) Keep only pair keys once
    pairs_keys = pairs_m[["br_idx", "mma_idx"]].drop_duplicates()
    
    # 3) Build lookup tables with attributes and lat/lon
    br_ll = gdf_purple_ll.copy()
    br_ll["br_idx"] = br_ll.index
    br_info = br_ll.assign(
        br_lat=br_ll.geometry.y,
        br_lon=br_ll.geometry.x,
        br_UF = br_ll["UF"] if "UF" in br_ll.columns else pd.NA,
        br_ID_OEMA = br_ll["ID_OEMA"] if "ID_OEMA" in br_ll.columns else pd.NA
    )[["br_idx","br_UF","br_ID_OEMA","br_lat","br_lon"]]
    
    mma_ll = gdf_mma_ll.copy()
    mma_ll["mma_idx"] = mma_ll.index
    mma_info = mma_ll.assign(
        mma_lat=mma_ll.geometry.y,
        mma_lon=mma_ll.geometry.x,
        mma_UF = mma_ll["UF"] if "UF" in mma_ll.columns else pd.NA,
        mma_ID_OEMA = mma_ll["ID_OEMA"] if "ID_OEMA" in mma_ll.columns else pd.NA
    )[["mma_idx","mma_UF","mma_ID_OEMA","mma_lat","mma_lon"]]
    
    # 4) Final pairs table
    pairs = pairs_keys.merge(br_info, on="br_idx", how="left").merge(mma_info, on="mma_idx", how="left")
    
    # Optional: distance in meters
    br_geom_m  = gdf_purple_m.geometry
    mma_geom_m = gdf_mma_m.geometry
    pairs["distance_m"] = pairs.apply(lambda r: br_geom_m[r.br_idx].distance(mma_geom_m[r.mma_idx]), axis=1).round(2)
    
    #print(pairs.head())

    return m, pairs

m, pairs = plot_purple_mma_map(df_purple, df_mma, draw_buffers=True)  
print(pairs)

In [ ]:
pairs.groupby(["br_UF"]).count()

In [ ]:
pairs.groupby("br_UF")["br_lat"].nunique()

In [ ]:
# base = Path.cwd().parent  # .../RQAR_2025_book
# out_dir = base / "data" / "DADOS_ESTACOES"
# out_dir.mkdir(parents=True, exist_ok=True)

# out_file = out_dir / "ParesIndicativas.csv"
# pairs.to_csv(out_file, index=False, encoding="utf-8")
# print("Saved to:", out_file.resolve())

In [ ]:
m

##### Criar novo dataframe único com informações do MMA e PurpleAir

In [ ]:
# Garantir que os pares são únicos 
pairs_u = (pairs.drop_duplicates(subset=["br_lat"]))
pairs_u.count()

In [ ]:
def unify_purple_mma(df_purple, df_mma, pairs_u):

    # ordem de colunas: df_purple primeiro, depois extras de df_mma
    out_cols = list(df_purple.columns) + [c for c in df_mma.columns if c not in df_purple.columns]

    # manter apenas índices que existem nos DFs atuais
    pairs_u = pairs_u[pairs_u["br_idx"].isin(df_purple.index) & pairs_u["mma_idx"].isin(df_mma.index)].copy()

    br_matched  = set(pairs_u["br_idx"])
    mma_matched = set(pairs_u["mma_idx"])

    # construir a linha mesclada de cada par: usa BR, senão MMA (não sobrescreve se ambos têm)
    merged_rows = []
    for _, pr in pairs_u.iterrows():
        br_row  = df_purple.loc[pr.br_idx]
        mma_row = df_mma.loc[pr.mma_idx]
        merged  = br_row.combine_first(mma_row)  # preenche só quando BR está NA
        merged  = merged.reindex(out_cols)
        merged["RECONHECIDA"] = "Sim"
        merged_rows.append(merged)

    merged_pairs_df = pd.DataFrame(merged_rows, columns=out_cols + ["RECONHECIDA"])

    # linhas sem par (dos dois lados), mantidas como estão
    br_un  = df_purple.loc[~df_purple.index.isin(br_matched)].reindex(columns=out_cols)
    br_un["RECONHECIDA"] = "Não"

    mma_un = df_mma.loc[~df_mma.index.isin(mma_matched)].reindex(columns=out_cols)
    mma_un["RECONHECIDA"] = ""

    # resultado final, mantendo a ordem de colunas
    df_final = pd.concat([merged_pairs_df, br_un, mma_un], ignore_index=True)
    
    return df_final[out_cols + ["RECONHECIDA"]]

In [ ]:
merged_dfs = unify_purple_mma(df_purple, df_mma, pairs_u)
merged_dfs

In [ ]:
def explode_pollutants(df_purple, col="POLUENTE"):
    out = df_purple.copy()

    # turn "VOC,PM1, PM25 ,PM10" into ["VOC","PM1","PM25","PM10"]
    out[col] = (
        out[col]
        .astype("string")
        .fillna("")
        .apply(lambda s: [p.strip() for p in s.split(",") if p.strip()])
    )

    # explode to one row per pollutant
    out = out.explode(col, ignore_index=True)

    return out.reset_index(drop=True)

df_merged_exp = explode_pollutants(merged_dfs)

In [ ]:
# base = Path.cwd().parent  # .../RQAR_2025_book
# out_dir = base / "data" / "DADOS_ESTACOES"
# out_dir.mkdir(parents=True, exist_ok=True)

# out_file = out_dir / "NEW_Monitoramento_QAr_BR.csv"
# df_merged_exp.to_csv(out_file, index=False, encoding="utf-8")
# print("Saved to:", out_file.resolve())

#### Importar série histórica de dados

In [ ]:
from dateutil import tz

In [ ]:
def month_ranges_2024():
    return [(datetime(2024,m,1,tzinfo=timezone.utc),
             datetime(2024,m+1,1,tzinfo=timezone.utc) if m<12 else datetime(2025,1,1,tzinfo=timezone.utc))
            for m in range(1,13)]

history_fields = "pm2.5_atm"  
AVERAGE_MIN = 60

hist_frames = []
for sid in df_purple["sensor_index"].head(20):  # raise gradually
    for start, end in month_ranges_2024():
        try:
            h = fetch_sensor_history(int(sid), history_fields, start, end, average=AVERAGE_MIN)
            rows, cols = h.get("data", []), h.get("fields", [])
            if not rows:
                continue
            hdf = pd.DataFrame(rows, columns=cols)
            hdf["sensor_index"] = int(sid)
            hist_frames.append(hdf)
            time.sleep(0.4)
        except requests.HTTPError as e:
            print(f"Sensor {sid} {start:%Y-%m} skipped:", e.response.text)
            time.sleep(0.6)

hist = pd.concat(hist_frames, ignore_index=True) if hist_frames else pd.DataFrame()
print(hist.shape)

In [ ]:
hist

In [ ]:
def fetch_sensor_history(sensor_index, fields, start, end, average=10):
    params = {
        "fields": fields,
        "start_timestamp": int(start.timestamp()),
        "end_timestamp": int(end.timestamp()),
        "average": average
    }
    url = f"{BASE}/sensors/{sensor_index}/history"
    r = requests.get(url, headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [ ]:
import pandas as pd

In [ ]:
history_fields = "pm2.5_atm"  # no time_stamp here
AVERAGE_MIN = 60  # hourly averages

def month_ranges_2024():
    return [(datetime(2024, m, 1, tzinfo=timezone.utc),
             datetime(2024, m+1, 1, tzinfo=timezone.utc) if m < 12 else datetime(2025, 1, 1, tzinfo=timezone.utc))
            for m in range(1, 13)]

hist_frames = []
for sid in df["sensor_index"].head(3):
    for start, end in month_ranges_2024():
        try:
            h = fetch_sensor_history(int(sid), history_fields, start, end, average=AVERAGE_MIN)
            cols = h.get("fields", [])
            rows = h.get("data", [])
            if not rows:
                continue
            hdf = pd.DataFrame(rows, columns=cols)
            # The response already includes a timestamp column (often named time_stamp)
            hdf["sensor_index"] = int(sid)  # add ID yourself
            hist_frames.append(hdf)
            time.sleep(0.4)
        except requests.HTTPError as e:
            print(f"Sensor {sid} {start:%Y-%m} skipped:", getattr(e, "response", None).text if getattr(e, "response", None) else e)
            time.sleep(0.6)

In [ ]:
hist = pd.concat(hist_frames, ignore_index=True) if hist_frames else pd.DataFrame()
print(hist)